# Project: Image Compression with k-means and SVD 🖼️
Two unsupervised ideas, one photo. **k-means** compresses *colours* (keep 16 colours, store a 4-bit index per pixel). **SVD/PCA** compresses *structure* (keep the top-k singular vectors). You'll measure which gives better quality per bit, and check the Eckart–Young theorem numerically.

Topic: [[Clustering]], [[PCA]] · guide note: [[Project - Image Compression]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

img = load_sample_image("china.jpg").astype(float) / 255     # bundled with scikit-learn, 427×640×3
h, w, _ = img.shape
psnr = lambda a, b: 10 * np.log10(1 / np.mean((a - b) ** 2))
plt.imshow(img); plt.axis("off"); plt.title(f"original {h}×{w}"); plt.show()

## 1. Colour quantisation with k-means
Every pixel is a point in RGB space. One **Lloyd step**: assign each point to its nearest centre, then move each centre to the mean of its points (keep a centre that got no points where it was). Return `(new_centres, labels)`.

In [ ]:
def kmeans_step(X, C):
    # TODO 1: assignment (argmin squared distance) + update (cluster means)
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_step():
    X = np.array([[0, 0], [0, 1], [10, 10], [10, 11.0]]); C = np.array([[0, 5], [10, 5.0], [50, 50.0]])
    newC, lab = kmeans_step(X, C)
    return lab.tolist() == [0, 0, 1, 1] and np.allclose(newC, [[0, 0.5], [10, 10.5], [50, 50]])
check("lloyd step", _t_step, "Points go to the nearest centre, centres move to their means, the empty centre stays put.")

def kmeans(X, k, iters=15, seed=0):
    r = np.random.default_rng(seed); C = X[r.choice(len(X), k, replace=False)]
    for _ in range(iters): C, _ = kmeans_step(X, C)
    return C

pixels = img.reshape(-1, 3); results = {}
if ready("lloyd step"):
    sample = pixels[np.random.default_rng(0).choice(len(pixels), 5000, replace=False)]   # fit on a sample (fast)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    for a, k in zip(ax, (4, 16, 64)):
        C = kmeans(sample, k); lab = ((pixels[:, None] - C[None]) ** 2).sum(-1).argmin(1)
        q = C[lab].reshape(img.shape); results[("kmeans", k)] = psnr(img, q)
        a.imshow(q); a.axis("off"); a.set_title(f"{k} colours, PSNR {results[('kmeans', k)]:.1f} dB")
    plt.show()

## 2. Low-rank approximation with SVD
For each colour channel (a $427\times640$ matrix $A$), keep the top $k$ singular values: $A_k = U_k\Sigma_kV_k^\top$. **Eckart–Young:** no rank-$k$ matrix is closer to $A$ in Frobenius norm, and the error is exactly $\sqrt{\sum_{i>k}\sigma_i^2}$.

In [ ]:
def low_rank(A, k):
    # TODO 2: best rank-k approximation via np.linalg.svd(A, full_matrices=False)
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_svd():
    A = img[:, :, 0]; s = np.linalg.svd(A, compute_uv=False); Ak = low_rank(A, 20)
    return np.linalg.matrix_rank(Ak) == 20 and np.isclose(np.linalg.norm(A - Ak), np.sqrt((s[20:] ** 2).sum()))
check("low rank", _t_svd, "Rank must be k and the Frobenius error must equal sqrt(sum of the discarded σ²) (Eckart–Young).")
if ready("low rank"):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    for a, k in zip(ax, (5, 20, 80)):
        q = np.clip(np.stack([low_rank(img[:, :, c], k) for c in range(3)], -1), 0, 1); results[("svd", k)] = psnr(img, q)
        a.imshow(q); a.axis("off"); a.set_title(f"rank {k}, PSNR {results[('svd', k)]:.1f} dB")
    plt.show()
    s = np.linalg.svd(img[:, :, 0], compute_uv=False)
    plt.semilogy(s); plt.xlabel("index"); plt.ylabel("singular value"); plt.title("how fast the spectrum decays = how compressible"); plt.show()

## 3. Quality per bit
Count storage in bits: the original is 24 bits/pixel. **k-means:** the palette (`k × 3` bytes) + `⌈log2 k⌉` bits per pixel. **SVD:** 3 channels × rank × (h + w + 1) numbers × 32 bits (float32). Return `(original_bits, kmeans_bits, svd_bits)`.

In [ ]:
def storage_bits(h, w, k_colors, rank):
    # TODO 3: bits for the original, the k-means version and the rank-`rank` SVD version
    raise NotImplementedError  # TODO 3

In [ ]:
check("storage", lambda: storage_bits(10, 20, 16, 2) == (4800, 16 * 24 + 200 * 4, 3 * 2 * 31 * 32), "Toy 10×20: original 4800, k-means 384 + 800, SVD 3·2·31·32.")
if ready("lloyd step", "low rank", "storage"):
    for (kind, k), q in sorted(results.items()):
        orig, kb, sb = storage_bits(h, w, k, k)
        bits = kb if kind == "kmeans" else sb
        print(f"{kind:>6} k={k:<3} PSNR {q:5.1f} dB   size {bits / orig:6.1%} of original")
    print("→ Which method gives better quality for the same size? Why does SVD struggle with sharp edges?")

<details><summary>▶ Solution</summary>

```python
def kmeans_step(X, C):
    d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)
    labels = d.argmin(1)
    newC = np.array([X[labels == k].mean(0) if np.any(labels == k) else C[k] for k in range(len(C))])
    return newC, labels
```

```python
def low_rank(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return (U[:, :k] * s[:k]) @ Vt[:k]
```

```python
def storage_bits(h, w, k_colors, rank):
    n_pixels = h * w
    kmeans_bits = k_colors * 3 * 8 + n_pixels * int(np.ceil(np.log2(k_colors)))
    svd_bits = 3 * rank * (h + w + 1) * 32
    return n_pixels * 24, kmeans_bits, svd_bits
```
</details>

## Stretch goals
- Use your own photo (`plt.imread`) and a mushroom from [[Danish Mushroom Classifier]].
- SVD on 8×8 **patches** instead of whole channels: that's the idea behind JPEG (with a fixed DCT basis instead of a learned one).
- Fit k-means in **Lab** colour space (perceptually uniform). Does it look better at the same PSNR?